In [7]:
!pip install playwright asyncio

In [26]:
import asyncio
import base64
import logging
import socket
import re

from dataclasses import dataclass
from typing import Optional
from urllib.parse import urlparse, urlunparse

from playwright.async_api import async_playwright, Browser, Page, BrowserContext

logger = logging.getLogger(__name__)

@dataclass
class BrowserSnapshot:
    screenshot_base64: str
    html: str
    url: str
    console_logs: list[str]


class BrowserSession:
    """
    Connects to a running Chrome instance (launched via Puppeteer or directly)
    using the Chrome DevTools Protocol (CDP) remote debugging port.
    
    The Chrome instance is started separately (e.g. via chrome_session.js or
    the custom_startup.sh hook in the vibe session container).
    
    This class is purely the Python-side attachment — no browser is launched here.
    """

    def __init__(self, domain: str, port: int = 9222):
        self.domain_ip = self.get_container_ip(domain)
        self.cdp_url = f"http://{self.domain_ip}:{port}"
        print(f"[BrowserSession] CDP URL resolved: {self.cdp_url}")
        self._playwright = None
        self._browser: Optional[Browser] = None
        self._context: Optional[BrowserContext] = None
        self._page: Optional[Page] = None
        self._console_logs: list[str] = []

    def get_container_ip(self, hostname: str) -> Optional[str]:
        """Resolve a hostname to its IP address."""
        try:
            ip = socket.gethostbyname(hostname)
            print(f"[BrowserSession] Resolved '{hostname}' → '{ip}'")
            return ip
        except socket.gaierror as e:
            print(f"[BrowserSession] ❌ Could not resolve hostname: {hostname} ({e})")
            return None

    def _is_ip(self, hostname: str) -> bool:
        """Return True if hostname is already an IPv4 address."""
        ip_pattern = re.compile(r'^(\d{1,3}\.){3}\d{1,3}$')
        return bool(ip_pattern.match(hostname))

    # ------------------------------------------------------------------
    # Lifecycle
    # ------------------------------------------------------------------

    async def connect(self) -> None:
        """Attach to the running Chrome instance via CDP."""
        logger.info(f"[BrowserSession] Connecting to CDP at {self.cdp_url}")
        print(f"[BrowserSession] Connecting to CDP at {self.cdp_url}")
        self._playwright = await async_playwright().start()
        self._browser = await self._playwright.chromium.connect_over_cdp(self.cdp_url)

        # Use first existing context (the one Puppeteer created)
        contexts = self._browser.contexts
        self._context = contexts[0] if contexts else await self._browser.new_context()

        # Use first existing page or open a new one
        pages = self._context.pages
        self._page = pages[0] if pages else await self._context.new_page()

        # Capture console output for LLM debug feedback
        self._page.on("console", lambda msg: self._console_logs.append(
            f"[{msg.type}] {msg.text}"
        ))

        logger.info(f"[BrowserSession] Connected. Current URL: {self._page.url}")
        print(f"[BrowserSession] ✅ Connected. Current URL: {self._page.url}")

    async def disconnect(self) -> None:
        """Detach from CDP without closing the browser."""
        if self._browser:
            await self._browser.close()   # closes CDP connection, not the browser process
        if self._playwright:
            await self._playwright.stop()
        logger.info("[BrowserSession] Disconnected from CDP.")

    # ------------------------------------------------------------------
    # Browser Actions
    # ------------------------------------------------------------------

    async def navigate(self, url: str) -> None:
        """Navigate the browser to a URL."""
        await self._page.goto(url, wait_until="networkidle")
        logger.info(f"[BrowserSession] Navigated to {url}")

    async def reload(self) -> None:
        """Reload the current page (useful after hot reload to force refresh)."""
        await self._page.reload(wait_until="networkidle")

    async def wait_for_hot_reload(self, timeout_ms: int = 5000) -> None:
        """
        Wait for the page network to settle after a hot reload.
        Assumes the dev server triggers a DOM update within timeout.
        """
        try:
            await self._page.wait_for_load_state("networkidle", timeout=timeout_ms)
        except Exception:
            logger.warning("[BrowserSession] Hot reload wait timed out — proceeding anyway")

    # ------------------------------------------------------------------
    # Capture Methods
    # ------------------------------------------------------------------

    async def screenshot(self, selector: Optional[str] = None) -> str:
        """
        Capture a screenshot. 
        If selector is given, captures only that element.
        Returns base64-encoded PNG string.
        """
        if selector:
            element = await self._page.query_selector(selector)
            if element:
                raw = await element.screenshot(type="png")
            else:
                logger.warning(f"[BrowserSession] Selector '{selector}' not found — full page screenshot")
                raw = await self._page.screenshot(type="png", full_page=True)
        else:
            raw = await self._page.screenshot(type="png", full_page=True)

        return base64.b64encode(raw).decode("utf-8")

    async def get_html(self, selector: Optional[str] = None) -> str:
        """
        Dump the rendered HTML.
        If selector given, returns innerHTML of that element only.
        """
        if selector:
            return await self._page.inner_html(selector)
        return await self._page.content()

    async def get_computed_styles(self, selector: str) -> dict:
        """Extract computed CSS styles for a given element."""
        return await self._page.eval_on_selector(
            selector,
            """el => {
                const styles = window.getComputedStyle(el);
                const result = {};
                for (let i = 0; i < styles.length; i++) {
                    const prop = styles[i];
                    result[prop] = styles.getPropertyValue(prop);
                }
                return result;
            }"""
        )

    async def capture_context(self, selector: Optional[str] = None) -> BrowserSnapshot:
        """
        Full context capture for the LLM:
        screenshot + HTML + current URL + console logs.
        
        This is the main method called by chat_mode.py before each LLM turn.
        """
        screenshot_b64 = await self.screenshot(selector)
        html = await self.get_html(selector)
        url = self._page.url

        snapshot = BrowserSnapshot(
            screenshot_base64=screenshot_b64,
            html=html,
            url=url,
            console_logs=list(self._console_logs),
        )

        # Clear log buffer after capture
        self._console_logs.clear()

        return snapshot

    async def after_vibe_commit(self, selector: Optional[str] = None) -> BrowserSnapshot:
        """
        Called by chat_mode.py after each vibe git commit.
        Waits for hot reload then captures fresh context for the LLM.
        """
        await self.wait_for_hot_reload()
        return await self.capture_context(selector)

    async def scroll_to(self, selector: str) -> None:
        """Scroll element into view."""
        await self._page.eval_on_selector(selector, "el => el.scrollIntoView()")

    async def resize_viewport(self, width: int, height: int) -> None:
        """Resize viewport for responsive testing."""
        await self._page.set_viewport_size({"width": width, "height": height})

    async def highlight_element(self, selector: str) -> None:
        """Visually highlight an element (useful for customer feedback)."""
        await self._page.eval_on_selector(
            selector,
            """el => {
                el.style.outline = '3px solid #ff4081';
                el.style.outlineOffset = '2px';
                setTimeout(() => {
                    el.style.outline = '';
                    el.style.outlineOffset = '';
                }, 2000);
            }"""
        )

In [27]:
session = BrowserSession(domain="codx-junior-default-workspace", port=9223)

[BrowserSession] Resolved 'codx-junior-default-workspace' → '172.18.0.2'
[BrowserSession] CDP URL resolved: http://172.18.0.2:9223


In [28]:
await session.connect()

[BrowserSession] Connecting to CDP at http://172.18.0.2:9223
[BrowserSession] ✅ Connected. Current URL: about:blank


In [29]:
await session.navigate("http://localhost:8008")